In [47]:
import pandas as pd
import os
from pathlib import Path

In [48]:
#BASE_DIR = Path(__file__).resolve().parent #For .py
BASE_DIR = Path.cwd()

In [49]:
def get_mapping():
    mapping = {
        1: ["de uitvoerder", "functioneel", "duidelijk", "focus op taak", "fijn teamlid", "taakgericht", "is een betrouwbare uitvoerder"],
        2: ["de doorvrager", "productie-waardig", "gebruikersgericht", "focus op waarde", "de verbinder", "projectgericht", "de kritische partner"],
        3: ["de systeemdenker", "de standaard", "richtinggevend", "focus op keten", "de kartrekker", "toekomstgericht", "de expert"],
        4: ["is een project-eigenaar"],
        0: ["heeft sturing nodig"]
    }

    # Gesplitst in Peer en Opdrachtgever
    peer_ids = {
        'zEFBRre0JvuV': "Analyse & Probleemoplossing",
        'xrpnryUhvlwC': "Kwaliteit & Vakmanschap",
        'tNNOU1qabV2K': "Visualisatie & Verhaal",
        'b4j7QEttWAM0': "Business & Context",
        'PbZsX4a9wSCB': "Samenwerken & Communicatie",
        'fGbuwROaGceQ': "Eigenaarschap & Initiatief"
    }
    
    client_ids = {
        'YeoMmpNa3umy': "Waarde & Vakmanschap",
        'swelQCdUyu9d': "Eigenaarschap & Resultaat",
        'vQmeObrQ7BdE': "Communicatie & Invloed"
    }
    
    return mapping, peer_ids, client_ids

In [50]:
def get_score(text, mapping):
    if pd.isna(text): 
        return None
    text = str(text).lower().strip()
    for key, values in mapping.items():
        if any(text.startswith(v) for v in values):
            return key
    return None

In [51]:
def process_to_row(df_subset, id_mapping, mapping_scores):
    """Hulpmethode om van een subset data één rij met datum-index te maken"""
    if df_subset.empty:
        return pd.DataFrame()
    
    # Verkrijg scores
    df_subset = df_subset.copy()
    df_subset["score"] = df_subset["answer_value"].apply(lambda x: get_score(x, mapping_scores))
    
    # Maak dictionary: {ID: Score}
    row_dict = dict(zip(df_subset["question_id"], df_subset["score"]))
    
    # Pak de datum van deze specifieke set
    date_val = pd.to_datetime(df_subset["submitted_at"]).dt.date.iloc[0]
    
    # Omzetten naar DF met hernoemde kolommen
    row_df = pd.DataFrame([row_dict], index=[date_val])
    row_df = row_df.rename(columns=id_mapping)
    row_df.index.name = "Datum" 
    return row_df

In [52]:
def save_to_excel(df, sheet_name, filename):
    """Slaat data op of voegt het toe aan een bestaand tabblad"""
    if os.path.exists(filename):
        with pd.ExcelWriter(filename, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
            try:
                # Probeer bestaande data te lezen om nieuwe rij eronder te plakken
                existing_df = pd.read_excel(filename, sheet_name=sheet_name, index_col=0)
                combined_df = pd.concat([existing_df, df])
                # Verwijder eventuele dubbele rijen op basis van datum
                combined_df = combined_df[~combined_df.index.duplicated(keep='last')]
                combined_df.to_excel(writer, sheet_name=sheet_name)
            except Exception:
                # Als sheet nog niet bestaat in het bestand
                df.to_excel(writer, sheet_name=sheet_name)
    else:
        # Nieuw bestand maken
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name=sheet_name)

In [ ]:
def run_pipeline(path1, path2):
    # 1. Data ophalen
    mapping_scores, peer_mapping, client_mapping = get_mapping()
    df1 = pd.read_parquet(path1, engine="pyarrow")
    df2 = pd.read_parquet(path2, engine="pyarrow")
    full_df = pd.concat([df1, df2], ignore_index=True)

    # 2. Medewerker naam bepalen voor bestandsnaam
    emp_name = full_df["employee_name"].iloc[0]
    filename = f"{emp_name}.xlsx"

    # 3. Splitsen en verwerken
    peer_data = full_df[full_df["question_id"].isin(peer_mapping.keys())]
    client_data = full_df[full_df["question_id"].isin(client_mapping.keys())]

    peer_row = process_to_row(peer_data, peer_mapping, mapping_scores)
    client_row = process_to_row(client_data, client_mapping, mapping_scores)

    # 4. Opslaan in Excel (met losse tabs)
    if not peer_row.empty:
        save_to_excel(peer_row, "data/Peer Reviews", filename)
    if not client_row.empty:
        save_to_excel(client_row, "data/Opdrachtgever Reviews", filename)

    print(f"Data succesvol verwerkt voor {emp_name} in {filename}")

# Uitvoeren
path1 = BASE_DIR / "data/transformed_responses/Peer_Feedback/peer_1.parquet"
path2 = BASE_DIR / "data/transformed_responses/Opdrachtgever_Feedback/opdrachtgever_1.parquet"
run_pipeline(path1, path2)

Data succesvol verwerkt voor employee_name in employee_name.xlsx
